# 07_diabetes_extension

Replicate the pipeline for **type-2 diabetes** new-onset. Because diabetes
incidence is lower than hypertension, the prospective risk model uses a
**2-year prediction window** (t -> t+2) to accumulate events. The Step 4
longitudinal verification (BMI reduction -> subsequent onset) reuses the
3-wave structure.

Key contrast with hypertension: in the diabetes risk model, regular exercise
is a *significant* protective factor (unlike hypertension), consistent with
the known role of physical activity in insulin sensitivity. Yet the Step 4
causal check remains null (protective direction, CI includes 1), reinforcing
that association in the risk model does not translate into a verified causal
effect of self-reported BMI change.

In [1]:
# 07_diabetes_extension.ipynb
# Diabetes new-onset: 2-year-window risk model + Step 4 verification.

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
TAB  = os.path.join(ROOT, "results", "tables")

panel = pd.read_parquet(os.path.join(DATA, "khp_panel_long.parquet"))
adult = panel[panel.age >= 19].copy()

In [2]:
# Prospective risk model with a 2-year window (t -> t+2).
PAIRS2 = [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
rows = []
for t0, t2 in PAIRS2:
    a = adult[adult.year==t0][["PIDWON","DM","BMI","age","SEX","smoke_cur","exer_reg","walk_days"]]
    b = panel[panel.year==t2][["PIDWON","DM"]]
    m = a.merge(b, on="PIDWON", suffixes=("", "_t2"))
    r = m[m["DM"] == 0].copy()
    r["incident"] = (r["DM_t2"] == 1).astype(int)
    r["t0y"] = t0
    rows.append(r)
D = pd.concat(rows, ignore_index=True).dropna(subset=["BMI","age","smoke_cur","exer_reg"])
D["female"] = (D["SEX"] == 2).astype(int)
D.to_parquet(os.path.join(DATA, "dm_analysis.parquet"))
print(f"DM 2-yr at-risk: {len(D)}, new onset {D['incident'].sum()} "
      f"({D['incident'].mean()*100:.2f}%)")

m = smf.logit("incident ~ BMI + age + female + smoke_cur + exer_reg + C(t0y)",
              data=D).fit(disp=0)
OR = np.exp(m.params); ci = np.exp(m.conf_int())
dm_risk = pd.DataFrame({
    "Variable": ["BMI (+1)","Age (+1yr)","Female","Current smoker","Regular exercise"],
    "OR":     [OR[v] for v in ["BMI","age","female","smoke_cur","exer_reg"]],
    "CI_low": [ci.loc[v,0] for v in ["BMI","age","female","smoke_cur","exer_reg"]],
    "CI_high":[ci.loc[v,1] for v in ["BMI","age","female","smoke_cur","exer_reg"]],
    "p":      [m.pvalues[v] for v in ["BMI","age","female","smoke_cur","exer_reg"]],
}).round(3)
dm_risk.to_csv(os.path.join(TAB, "table6_dm_risk_model.csv"), index=False)
print(dm_risk.to_string(index=False))

DM 2-yr at-risk: 31263, new onset 739 (2.36%)
        Variable    OR  CI_low  CI_high     p
        BMI (+1) 1.154   1.131    1.178 0.000
      Age (+1yr) 1.035   1.029    1.040 0.000
          Female 0.845   0.720    0.990 0.038
  Current smoker 1.250   1.007    1.551 0.043
Regular exercise 0.844   0.727    0.979 0.025


In [3]:
# Step 4 for diabetes: BMI reduction (t0->t1) -> DM new-onset at t2.
TRIPLES = [(2019,2020,2021),(2020,2021,2022),(2021,2022,2023),(2022,2023,2024)]
rows = []
for t0, t1, t2 in TRIPLES:
    d0 = adult[adult.year==t0][["PIDWON","BMI","DM","age","SEX","smoke_cur","exer_reg"]]
    d1 = adult[adult.year==t1][["PIDWON","BMI","DM"]]
    d2 = adult[adult.year==t2][["PIDWON","DM"]]
    m2 = (d0.merge(d1,on="PIDWON",suffixes=("_0","_1"))
            .merge(d2,on="PIDWON").rename(columns={"DM":"DM_2"}))
    r = m2[(m2["DM_0"]==0)&(m2["DM_1"]==0)].dropna(subset=["BMI_0","BMI_1","DM_2"]).copy()
    r["incident_t2"] = (r["DM_2"]==1).astype(int)
    r["achieved"]    = (r["BMI_1"]-r["BMI_0"] <= -1.0).astype(int)
    r["t0y"] = t0
    rows.append(r)
S = pd.concat(rows, ignore_index=True)
S["female"] = (S["SEX"]==2).astype(int)
S = S.dropna(subset=["smoke_cur","exer_reg","BMI_0","age"])
S.to_parquet(os.path.join(DATA, "dm_step4_data.parquet"))

crude = S.groupby("achieved")["incident_t2"].mean() * 100
mm = smf.logit("incident_t2 ~ achieved + BMI_0 + age + female + smoke_cur + exer_reg + C(t0y)",
               data=S).fit(disp=0)
OR = np.exp(mm.params["achieved"]); ci = np.exp(mm.conf_int().loc["achieved"])
dm_s4 = pd.DataFrame({
    "Metric": ["Crude achieved %","Crude not %","Adjusted OR","CI_low","CI_high","p","Achieved N"],
    "Value":  [round(crude[1],2), round(crude[0],2), round(OR,3),
               round(ci[0],3), round(ci[1],3), round(mm.pvalues["achieved"],3),
               int(S["achieved"].sum())],
})
dm_s4.to_csv(os.path.join(TAB, "table7_dm_step4.csv"), index=False)
print(dm_s4.to_string(index=False))
print("\nDiabetes Step 4: protective direction (OR<1) but CI includes 1 -> null,"
      " consistent with hypertension.")

          Metric    Value
Crude achieved %    1.580
     Crude not %    1.330
     Adjusted OR    0.909
          CI_low    0.662
         CI_high    1.249
               p    0.555
      Achieved N 3047.000

Diabetes Step 4: protective direction (OR<1) but CI includes 1 -> null, consistent with hypertension.
